In [ ]:
import torch
import os
import torch.nn.functional as F


folder_kv_base = 'sims_kv_test'
filename = 'batch_0_k_previous.pt'
path_kv_file = os.path.join(folder_kv_base, filename)

len_prompt = 128
num_block = 4
len_target = 256
size_block = int(len_target / num_block)



In [18]:
def token_nonsimilarity_score(
    sim: torch.Tensor,
    p: float = 3.0,
) -> torch.Tensor:
    """
    sim: shape [steps, layers, tokens], values in [0, 1]
    p: power mean exponent, larger => more sensitive to low-similarity cases

    Returns:
        score: shape [tokens], larger means more non-similar
    """
    assert sim.ndim == 3, f"Expected 3D tensor [steps, layers, tokens], got shape {tuple(sim.shape)}"

    # Convert similarity to non-similarity / change magnitude
    diff = 1.0 - sim

    # Power mean over step and layer
    # First raise to power p, then average over dims 0 and 1, then take 1/p power
    score = diff.pow(p).mean(dim=(0, 1)).pow(1.0 / p)

    return score


In [ ]:
matrix_sim_step_layer_token = torch.load(path_kv_file)
matrix_sim_step_layer_token = F.pad(matrix_sim_step_layer_token, (0,0,0,0,1,0), value=1.0)

In [ ]:
list_idx_sim_sorted = []

for id_block in range(num_block):
    position_start = len_prompt + size_block * id_block
    matrix_sim_step_layer_token_cached = matrix_sim_step_layer_token[:size_block, :, :position_start]  #(steps_block, len_cached)
    score_diff_token = token_nonsimilarity_score(matrix_sim_step_layer_token_cached)
    idx_diff_token_decending = torch.argsort(score_diff_token, descending=True)
    list_idx_sim_sorted.append({'filename': filename, 'block': {id_block}, 'idx': idx_diff_token_decending.tolist(), 'value': score_diff_token[idx_diff_token_decending].tolist()})
# end

tensor([0.0025, 0.0013, 0.0012,    nan,    nan, 0.0025, 0.0011, 0.0047, 0.0849,
        0.0023,    nan, 0.0042,    nan,    nan, 0.0019, 0.0023,    nan, 0.0030,
           nan, 0.0014,    nan,    nan, 0.0117, 0.0029,    nan,    nan,    nan,
           nan, 0.0012,    nan, 0.0156, 0.0009,    nan,    nan,    nan, 0.0016,
           nan, 0.0126, 0.0029,    nan,    nan, 0.0042, 0.0010, 0.0120, 0.0043,
        0.0042, 0.0195, 0.0029, 0.0020, 0.0020, 0.0474, 0.0038, 0.0077, 0.0038,
           nan, 0.0064, 0.0029, 0.0032, 0.0009, 0.0017, 0.0566, 0.0040,    nan,
        0.0589, 0.0034, 0.0085,    nan,    nan, 0.0027, 0.0228, 0.0045, 0.0047,
        0.0047, 0.0052, 0.0148, 0.0010, 0.0086, 0.0041, 0.0022,    nan, 0.0382,
        0.0010, 0.0004,    nan, 0.0031, 0.0033,    nan,    nan, 0.1351, 0.0107,
        0.0506, 0.0010,    nan, 0.0137, 0.0025,    nan, 0.0372, 0.0252, 0.0015,
           nan,    nan, 0.0699, 0.0021,    nan, 0.0341,    nan, 0.0014, 0.0019,
        0.0019, 0.0023, 0.0021, 0.0530, 

In [23]:
list_idx_sim_sorted[0]['value']

[nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 nan,
 0.13507242500782013,
 0.08493614941835403,
 0.0699421763420105,
 0.06480773538351059,
 0.058880094438791275,
 0.056579768657684326,
 0.053010646253824234,
 0.05060700327157974,
 0.04739385098218918,
 0.038163915276527405,
 0.03723873198032379,
 0.034141141921281815,
 0.02515874058008194,
 0.0228325966745615,
 0.019536612555384636,
 0.015622931532561779,
 0.014800834469497204,
 0.013738037087023258,
 0.012629176490008831,
 0.012018118984997272,
 0.011706726625561714,
 0.010721464641392231,
 0.008551226928830147,
 0.008544853888452053,
 0.007682557217776775,
 0.007419895380735397,
 0.006447909399867058,
 0.005155032034963369,
 0.004737325478345156,
 0.004706432577222586,
 0.004689730703830719,
 0.004492306150496006,
 0.004297983832657337,
 0.004234557040035725,
 